# Preprocess `adata` into gene list files

This notebook loads an `.h5ad` file, extracts:
- all gene names (`gene_names.txt`)
- interest genes (`interest_genes.txt`)

Both outputs are saved in the `data/` directory.

In [3]:
from pathlib import Path
import numpy as np
import scanpy as sc

# Resolve project root robustly whether notebook runs from repo root or notebooks/
cwd = Path.cwd()
if (cwd / "data").exists():
    project_root = cwd
elif (cwd.parent / "data").exists():
    project_root = cwd.parent
else:
    raise FileNotFoundError("Could not find a `data/` directory from current working directory.")

data_dir = project_root / "data/raw"
h5ad_path = data_dir / "SERGIO_sim_DS9.h5ad"
gene_names_out = data_dir / "gene_names.txt"
interest_genes_out = data_dir / "interest_genes.txt"

print(f"Project root: {project_root}")
print(f"Input h5ad: {h5ad_path}")

Project root: /nfs/roberts/project/pi_sk2433/sv496/RiTINI
Input h5ad: /nfs/roberts/project/pi_sk2433/sv496/RiTINI/data/raw/SERGIO_sim_DS9.h5ad


In [4]:
adata = sc.read_h5ad(h5ad_path)
print(adata)

# 1) All genes from var_names
gene_names = adata.var_names.astype(str).tolist()

# 2) Interest genes:
#    Prefer explicit annotation columns in adata.var if available;
#    otherwise fallback to top-variance genes from adata.X.
candidate_interest_columns = [
    "interest", "is_interest", "interest_gene", "interest_genes",
    "gene_of_interest", "genes_of_interest", "highly_variable"
]

interest_genes = []
for col in candidate_interest_columns:
    if col in adata.var.columns:
        mask = adata.var[col]
        if hasattr(mask, "astype"):
            mask = mask.astype(bool)
        selected = adata.var_names[mask].astype(str).tolist()
        if len(selected) > 0:
            interest_genes = selected
            print(f"Using adata.var['{col}'] for interest genes ({len(interest_genes)} genes).")
            break

if len(interest_genes) == 0:
    top_n = min(20, adata.n_vars)
    x = adata.X
    x_np = x.toarray() if hasattr(x, "toarray") else np.asarray(x)
    variances = np.var(x_np, axis=0)
    top_idx = np.argsort(variances)[-top_n:][::-1]
    interest_genes = [gene_names[i] for i in top_idx]
    print(f"No explicit interest-gene column found; using top {top_n} variable genes.")

# Ensure deterministic unique ordering
interest_genes = list(dict.fromkeys(interest_genes))

print(f"Total genes: {len(gene_names)}")
print(f"Interest genes: {len(interest_genes)}")

AnnData object with n_obs × n_vars = 900 × 100
    obs: 'sim_time_bin', 'sim_timepoint', 'pseudotime', 'pseudotime_bin'
    uns: 'log1p', 'pca'
    obsm: 'X_pca', 'X_phate', 'X_phate_pca'
    varm: 'PCs'
No explicit interest-gene column found; using top 20 variable genes.
Total genes: 100
Interest genes: 20


In [5]:
with open(gene_names_out, "w") as f:
    for gene in gene_names:
        f.write(f"{gene}\n")

with open(interest_genes_out, "w") as f:
    for gene in interest_genes:
        f.write(f"{gene}\n")

print(f"Saved gene names to: {gene_names_out}")
print(f"Saved interest genes to: {interest_genes_out}")
print("\nFirst 10 gene names:")
print(gene_names[:10])
print("\nFirst 10 interest genes:")
print(interest_genes[:10])

Saved gene names to: /nfs/roberts/project/pi_sk2433/sv496/RiTINI/data/raw/gene_names.txt
Saved interest genes to: /nfs/roberts/project/pi_sk2433/sv496/RiTINI/data/raw/interest_genes.txt

First 10 gene names:
['G0', 'G1', 'G2', 'G3', 'G4', 'G5', 'G6', 'G7', 'G8', 'G9']

First 10 interest genes:
['G59', 'G72', 'G0', 'G7', 'G99', 'G8', 'G78', 'G11', 'G81', 'G36']
